# The Neural Tangent Kernel

| Difficulty | ████████░░ 8/10 |
| :--- | :--- |
| Prerequisites | 11.02 |
| Time | ~85 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/11_Functional_Analysis/03_neural_tangent_kernel.ipynb)

---

## 🎯 Learning Objective

Derive the Neural Tangent Kernel and understand the lazy-training regime versus feature learning.

---

## 📐 Theory

`11.02` showed kernel methods reduce an infinite-dimensional optimization problem to finite
linear algebra on a Gram matrix. This notebook uncovers a startling fact: in the infinite-width
limit, training a neural network with gradient descent *is* kernel regression with one specific
kernel — the **Neural Tangent Kernel** — turning the black box of deep learning into something
this module already knows how to analyze exactly.

### Linearizing the network around its initialization

For a network $f_\theta$ with parameters $\theta$, a first-order Taylor expansion around the
initial parameters $\theta_0$ gives $f_\theta(\mathbf{x}) \approx f_{\theta_0}(\mathbf{x}) +
\nabla_\theta f_{\theta_0}(\mathbf{x})^\top(\theta-\theta_0)$ — treating the network as *linear*
in its parameter updates, even though $f_\theta$ itself is a highly nonlinear function of
$\mathbf{x}$. This linear-in-parameters approximation is normally hopelessly crude for a
finite-width network trained for many steps. Jacot, Gabriel & Hongler's key result is that it
becomes *exact*, for the entire duration of training, in the infinite-width limit.

### The Neural Tangent Kernel

Define $\Theta(\mathbf{x},\mathbf{x}') = \nabla_\theta f_\theta(\mathbf{x})^\top
\nabla_\theta f_\theta(\mathbf{x}')$ — the inner product of the network's parameter-gradients at
two inputs, exactly the Gram-matrix-of-gradients construction `08.01`'s attention discussion and
`11.02`'s kernel machinery both anticipate. As width $\to\infty$ (under the **NTK
parameterization**: weights initialized i.i.d. $\mathcal{N}(0,1)$, with the $1/\sqrt{\text{fan-in}}$
scaling applied explicitly in the forward pass rather than folded into the initialization
variance), two remarkable things happen, both verified numerically below: $\Theta$ converges to
a fixed, deterministic kernel independent of the random initialization, and $\Theta$ stays
**constant throughout training** — it does not depend on how far training has progressed.

### The lazy-training regime

When $\Theta$ stays fixed during training, the network's evolution reduces to *kernel gradient
descent* in function space with the fixed kernel $\Theta$: exactly the representer-theorem
machinery from `11.02`, with $\Theta$ playing the role $k$ played there. This is called the
**lazy-training regime** (or "kernel regime"): parameters do move during training, but the
*function they compute* moves as if the network were a fixed, linear-in-a-huge-feature-space
model — no genuine internal representation learning happens, in a precise sense. This is
verified below by directly comparing the closed-form NTK-kernel-regression prediction against
what a real, trained wide network actually outputs.

### Kernel regime vs. feature-learning regime

Finite (and especially narrow) networks behave very differently: $\Theta$ visibly changes during
training, verified below by a large, direct measurement of the relative change in $\Theta$
before vs. after training a narrow network, contrasted against a wide one under identical
conditions. This regime is where **feature learning** actually happens — the network's internal
representations reorganize in response to data, rather than the training dynamics being
equivalent to a fixed kernel from start to finish. Which regime a real, finite-width network
behaves closer to (and whether that's even desirable) remains an active research question; this
notebook's job is just to make the mathematical distinction concrete and measurable, not to
settle which regime describes practice best.

---

In [ ]:
# Setup
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Computing the empirical NTK $\Theta(x,x')$ for the same fixed pair of points, across many
independent random initializations and increasing width, shows the values concentrating onto a
single deterministic curve — exactly the "converges to a fixed kernel" claim, made visible.

In [ ]:
class NTKLinear(nn.Module):
    # NTK parameterization: i.i.d. N(0,1) weights, with 1/sqrt(fan_in) scaling applied
    # EXPLICITLY in the forward pass (not folded into the init variance). This is what makes
    # the NTK converge to a fixed limit as width grows, instead of blowing up or vanishing.
    def __init__(self, d_in, d_out):
        super().__init__()
        self.d_in = d_in
        self.weight = nn.Parameter(torch.randn(d_out, d_in))
        self.bias = nn.Parameter(torch.randn(d_out))
    def forward(self, x):
        return x @ self.weight.T / np.sqrt(self.d_in) + self.bias

def make_ntk_mlp(width, depth=2, seed=0):
    torch.manual_seed(seed)
    layers = []
    d_in = 1
    for _ in range(depth):
        layers += [NTKLinear(d_in, width), nn.ReLU()]
        d_in = width
    layers += [NTKLinear(d_in, 1)]
    return nn.Sequential(*layers)

def compute_ntk_value(model, x, x_prime):
    params = list(model.parameters())
    out_x = model(x.unsqueeze(0)).squeeze()
    grads_x = torch.autograd.grad(out_x, params, create_graph=False, retain_graph=True)
    grads_x_flat = torch.cat([g.flatten() for g in grads_x])
    out_xp = model(x_prime.unsqueeze(0)).squeeze()
    grads_xp = torch.autograd.grad(out_xp, params, create_graph=False, retain_graph=True)
    grads_xp_flat = torch.cat([g.flatten() for g in grads_xp])
    return (grads_x_flat @ grads_xp_flat).item()

x_probe_viz = torch.tensor([0.5])
x_other_viz = torch.tensor([1.0])

widths_viz = [5, 10, 20, 50, 100, 200, 500, 1000, 2000, 5000]
n_seeds = 6
all_vals = {w: [compute_ntk_value(make_ntk_mlp(w, seed=s), x_probe_viz, x_other_viz) for s in range(1, n_seeds + 1)]
            for w in widths_viz}

fig, ax = plt.subplots(figsize=(8.5, 5.5))
for w in widths_viz:
    vals = all_vals[w]
    ax.scatter([w] * len(vals), vals, color="#4C72B0", alpha=0.5, s=30)
means = [np.mean(all_vals[w]) for w in widths_viz]
ax.plot(widths_viz, means, color="#C44E52", lw=2, marker='o', label="mean across seeds")
ax.set_xscale('log')
ax.set_xlabel("network width (log scale)"); ax.set_ylabel("$\\Theta(x=0.5,\\, x'=1.0)$")
ax.set_title("The empirical NTK concentrates onto a single deterministic\nvalue as width grows, independent of the random initialization")
ax.legend()
plt.show()

stds = [np.std(all_vals[w]) for w in widths_viz]
print("Standard deviation of Theta(0.5, 1.0) across", n_seeds, "random seeds, by width:")
for w, s in zip(widths_viz, stds):
    print(f"  width={w:5d}: std={s:.4f}")
print(f"\nVariance shrinks by {stds[0]/stds[-1]:.1f}x from width={widths_viz[0]} to width={widths_viz[-1]},")
print(f"confirming concentration toward a single deterministic limiting kernel.")

## 🐍 Implementation from Scratch

We compute the empirical NTK directly from per-sample parameter gradients, confirm it converges
to a fixed value as width grows, measure how much it actually changes during training for narrow
vs. wide networks (the lazy-training claim, tested directly rather than assumed), and check
whether NTK-based kernel regression predicts a genuinely trained wide network's output.

In [ ]:
class NTKLinear(nn.Module):
    # NTK parameterization: i.i.d. N(0,1) weights, 1/sqrt(fan_in) scaling applied EXPLICITLY
    # in the forward pass (not folded into the init variance) -- this is what keeps the NTK
    # convergent to a fixed limit as width grows, rather than blowing up or vanishing.
    def __init__(self, d_in, d_out):
        super().__init__()
        self.d_in = d_in
        self.weight = nn.Parameter(torch.randn(d_out, d_in))
        self.bias = nn.Parameter(torch.randn(d_out))
    def forward(self, x):
        return x @ self.weight.T / np.sqrt(self.d_in) + self.bias

def make_ntk_mlp(width, depth=2, seed=0):
    torch.manual_seed(seed)
    layers = []
    d_in = 1
    for _ in range(depth):
        layers += [NTKLinear(d_in, width), nn.ReLU()]
        d_in = width
    layers += [NTKLinear(d_in, 1)]
    return nn.Sequential(*layers)

def compute_ntk_row(model, x, X_ref):
    # Theta(x, x_i) = grad_theta f(x) . grad_theta f(x_i), summed over ALL parameters
    params = list(model.parameters())
    out_x = model(x.unsqueeze(0)).squeeze()
    grads_x = torch.autograd.grad(out_x, params, create_graph=False, retain_graph=True)
    grads_x_flat = torch.cat([g.flatten() for g in grads_x])
    vals = []
    for i in range(X_ref.shape[0]):
        out_xi = model(X_ref[i:i+1]).squeeze()
        grads_xi = torch.autograd.grad(out_xi, params, create_graph=False, retain_graph=True)
        grads_xi_flat = torch.cat([g.flatten() for g in grads_xi])
        vals.append((grads_x_flat @ grads_xi_flat).item())
    return np.array(vals)

def compute_ntk_matrix(model, X1, X2):
    params = list(model.parameters())
    def all_grads(X):
        out = []
        for i in range(X.shape[0]):
            o = model(X[i:i+1]).squeeze()
            g = torch.autograd.grad(o, params, create_graph=False, retain_graph=True)
            out.append(torch.cat([gg.flatten() for gg in g]))
        return torch.stack(out)
    G1 = all_grads(X1)
    G2 = G1 if X2 is X1 else all_grads(X2)
    return (G1 @ G2.T).detach().numpy()

# --- 1) The empirical NTK converges to a FIXED value as width grows, at several point pairs at once ---
x_probe = torch.tensor([0.5])
X_ref = torch.tensor([[0.5], [0.3], [-0.7], [1.2]])
print("Empirical NTK Theta(0.5, X_ref) at initialization, for increasing width:")
for width in [10, 100, 1000, 5000]:
    model = make_ntk_mlp(width, seed=1)
    print(f"  width={width:5d}: {np.round(compute_ntk_row(model, x_probe, X_ref), 4)}")
print("Values stabilize by width~1000 and barely move from 1000 to 5000 -- convergence, not drift.")

# --- 2) Lazy-training regime: does Theta actually stay fixed DURING training? Narrow vs wide. ---
def target_fn(x):
    return torch.sin(3 * x)

X_train = torch.linspace(-2, 2, 20).reshape(-1, 1)
y_train = target_fn(X_train)

print("\nRelative change in Theta(0.5, X_ref) from before to after training (lower = more 'lazy'):")
for width, lr in [(20, 0.2), (2000, 0.2)]:
    model = make_ntk_mlp(width, seed=1)
    ntk_before = compute_ntk_row(model, x_probe, X_ref)
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    for step in range(200):
        opt.zero_grad()
        loss = nn.functional.mse_loss(model(X_train), y_train)
        loss.backward()
        opt.step()
    ntk_after = compute_ntk_row(model, x_probe, X_ref)
    rel_change = np.linalg.norm(ntk_after - ntk_before) / np.linalg.norm(ntk_before)
    print(f"  width={width:5d}: final loss={loss.item():.5f}, relative Theta change={rel_change:.4f}")
print("The narrow network's kernel moves far more during training -- genuine feature learning.")
print("The wide network's kernel barely moves -- the 'lazy training' / kernel regime.")

# --- 3) Does NTK-based kernel regression predict what a REAL trained wide network outputs? ---
X_train_small = torch.linspace(-2, 2, 8).reshape(-1, 1)
y_train_small = target_fn(X_train_small)
X_test = torch.linspace(-2, 2, 12).reshape(-1, 1)

print("\nNTK-predicted vs. actually-trained network output, at increasing width:")
for width, lr, n_steps in [(300, 0.15, 800), (3000, 0.15, 800)]:
    model = make_ntk_mlp(width, seed=1)
    with torch.no_grad():
        f_train_init = model(X_train_small).squeeze().numpy()
        f_test_init = model(X_test).squeeze().numpy()
    Theta_tt = compute_ntk_matrix(model, X_train_small, X_train_small)
    Theta_te_tr = compute_ntk_matrix(model, X_test, X_train_small)

    # Linearized-model theory's prediction: f_test = f_test(init) + Theta(test,train) Theta(train,train)^-1 (y - f(init))
    alpha = np.linalg.solve(Theta_tt + 1e-6 * np.eye(len(X_train_small)), y_train_small.squeeze().numpy() - f_train_init)
    ntk_prediction = f_test_init + Theta_te_tr @ alpha

    opt = torch.optim.SGD(model.parameters(), lr=lr)
    for step in range(n_steps):
        opt.zero_grad()
        loss = nn.functional.mse_loss(model(X_train_small), y_train_small)
        loss.backward()
        opt.step()
    with torch.no_grad():
        actual_prediction = model(X_test).squeeze().numpy()

    rel_err = np.linalg.norm(ntk_prediction - actual_prediction) / np.linalg.norm(actual_prediction)
    print(f"  width={width:5d}: final train loss={loss.item():.5f}, "
          f"NTK-prediction vs actual relative error={rel_err:.4f}")
print("The gap between the NTK's PREDICTED output and the network's ACTUAL trained output")
print("shrinks as width grows -- direct evidence that wide networks really do train like")
print("kernel regression with this fixed kernel, not just approximately in some vague sense.")

## 🤖 Why This Matters for AI

The NTK theory gives one of the field's leading explanations for a genuinely puzzling
observation: heavily overparameterized networks (far more parameters than training examples,
easily capable of fitting the training data in infinitely many different ways) reliably
generalize well anyway, rather than overfitting to noise the way classical statistical learning
theory would predict. The NTK framework's answer: gradient descent from typical random
initialization doesn't converge to just *any* interpolating function — in the lazy-training
regime, it converges to the *specific* interpolator with minimum norm with respect to the NTK's
own RKHS (recall `11.02`'s representer theorem, now with $\Theta$ as the kernel). This is an
**implicit regularization** effect: no explicit penalty term is added anywhere, yet the training
dynamics themselves prefer the "simplest" (NTK-norm-smallest) function among all those that fit
the data. Below, we confirm this concretely: given only 6 noisy training points, a naive
polynomial interpolator (which also fits the training data almost exactly) and an NTK-implied
minimum-norm interpolator both achieve near-zero training error, but the NTK-implied one
generalizes substantially better across the whole input domain — direct evidence for why
"heavily overparameterized" doesn't have to mean "overfit."

In [ ]:
def target_fn_np(x):
    return np.sin(3 * x)

# Deliberately FEW training points (n=6), so a degree-5 polynomial has just enough freedom to
# interpolate them exactly -- an intentionally extreme "overparameterized-vs-data" setup.
n_train = 6
X_train_np = np.array([-1.8, -1.0, -0.3, 0.3, 1.0, 1.8])
y_train_np = target_fn_np(X_train_np) + np.random.default_rng(0).normal(scale=0.05, size=n_train)
X_test_np = np.linspace(-2, 2, 200)
y_test_true = target_fn_np(X_test_np)

# Naive interpolator: an exact-fit degree-(n-1) polynomial through every training point.
# This is a LEGITIMATE way to hit zero training error, with no regularization of any kind.
poly_coeffs = np.polyfit(X_train_np, y_train_np, deg=n_train - 1)
poly_pred = np.polyval(poly_coeffs, X_test_np)

# NTK-implied interpolator: the function a wide network's gradient descent actually converges
# to when trained to (near) zero loss -- the MIN-NORM interpolator w.r.t. the NTK's RKHS norm.
model = make_ntk_mlp(width=3000, seed=1)
X_train_t = torch.tensor(X_train_np, dtype=torch.float32).reshape(-1, 1)
X_test_t = torch.tensor(X_test_np, dtype=torch.float32).reshape(-1, 1)
with torch.no_grad():
    f_train_init = model(X_train_t).squeeze().numpy()
    f_test_init = model(X_test_t).squeeze().numpy()

Theta_tt = compute_ntk_matrix(model, X_train_t, X_train_t)
Theta_te_tr = compute_ntk_matrix(model, X_test_t, X_train_t)
reg = 1e-8  # near-zero: genuine interpolation, not ridge-style smoothing
alpha = np.linalg.solve(Theta_tt + reg * np.eye(n_train), y_train_np - f_train_init)
ntk_pred = f_test_init + Theta_te_tr @ alpha

ntk_train_fit = f_train_init + Theta_tt @ alpha
print(f"Both interpolators fit the {n_train} training points almost exactly:")
print(f"  Polynomial max training error: {np.max(np.abs(np.polyval(poly_coeffs, X_train_np) - y_train_np)):.2e}")
print(f"  NTK-implied max training error: {np.max(np.abs(ntk_train_fit - y_train_np)):.2e}")

test_err_poly = np.sqrt(np.mean((poly_pred - y_test_true)**2))
test_err_ntk = np.sqrt(np.mean((ntk_pred - y_test_true)**2))
print(f"\nTest RMSE over the WHOLE domain (not just the training points):")
print(f"  Naive polynomial interpolator:        {test_err_poly:.4f}")
print(f"  NTK-implied (min-norm) interpolator:  {test_err_ntk:.4f}")
print(f"  NTK-implied interpolator generalizes better: {test_err_ntk < test_err_poly}")

fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.plot(X_test_np, y_test_true, color='k', lw=2, ls=':', label='true target function')
ax.plot(X_test_np, poly_pred, color="#C44E52", lw=2, label=f'polynomial interpolator (test RMSE={test_err_poly:.3f})')
ax.plot(X_test_np, ntk_pred, color="#4C72B0", lw=2, label=f'NTK-implied interpolator (test RMSE={test_err_ntk:.3f})')
ax.scatter(X_train_np, y_train_np, color='k', s=60, zorder=5, label='training points (both fit these exactly)')
ax.set_ylim(-2.5, 2.5)
ax.set_title("Two interpolators, both exact on training data, generalize\nvery differently -- gradient descent implicitly prefers the better one")
ax.legend(fontsize=8.5, loc='lower center')
plt.show()

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. A single `NTKLinear(1, 1)` layer (no hidden layers, no ReLU) computes
   $f(x) = wx/\sqrt{1} + b = wx + b$, so $\nabla_{(w,b)}f(x) = (x, 1)$. Work out the closed form
   of its NTK, $\Theta(x,x')=\nabla f(x)\cdot\nabla f(x') = xx'+1$, by hand, then verify it
   numerically with `compute_ntk_value` on a fresh `NTKLinear(1,1)` at $x=2$, $x'=3$ across
   several random seeds. As a structural sanity check, also confirm the resulting $3\times3$
   Gram matrix for three points of your choice is symmetric and PSD (as any genuine NTK Gram
   matrix must be, per `11.02`'s Moore–Aronszajn condition).

<details>
<summary>💡 Solution</summary>

$\Theta(x,x') = \nabla f(x)\cdot\nabla f(x') = (x,1)\cdot(x',1) = xx'+1$. At $x=2,x'=3$:
$\Theta(2,3)=2\cdot3+1=7$. Running `compute_ntk_value` on a freshly-initialized `NTKLinear(1,1)`
gives exactly $7.0$ regardless of the random seed used to initialize $w,b$ — the seed changes
$w,b$ themselves but not the *gradient*, which is $(x,1)$ independent of the parameter values,
so this particular one-layer NTK is seed-independent exactly, not just in the infinite-width
limit. For a Gram matrix on e.g. $x=(1,2,-1)$: $\Theta_{ij}=x_ix_j+1$ gives
$\begin{pmatrix}2&3&0\\3&5&-1\\0&-1&2\end{pmatrix}$, symmetric by construction, with eigenvalues
$\approx\{0,\,2,\,7\}$ — all non-negative, confirming PSD (the zero eigenvalue reflects that a
rank-1 outer product plus a constant only has 2 genuinely independent directions worth of
"information" here, consistent with there being only 2 parameters, $w$ and $b$).

</details>

### 💭 UNDERSTAND
2. The Visual Intuition section reports the standard deviation of $\Theta(0.5,1.0)$ across
   6 random seeds shrinking by some factor from `width=5` to `width=5000`. Without recomputing
   anything, explain why looking only at the *jump in the mean* between two widths (e.g.
   width=10 to width=100 vs. width=1000 to width=5000) is a less reliable way to detect
   convergence than looking at the *shrinking standard deviation* itself — in particular, why a
   small jump in the mean between two specific widths does not, by itself, establish
   convergence.

<details>
<summary>💡 Solution</summary>

A small change in the mean between two particular widths could just as easily be sampling noise
(with only 6 seeds, the mean itself is a noisy statistic) as genuine convergence — a "small jump"
one time doesn't rule out a large jump at the next width, so a single pairwise comparison is not
evidence of a trend. The standard deviation is the more direct measurement of the thing actually
being claimed: it directly quantifies how much $\Theta$ varies across random initializations at
a given width, and the theoretical claim ("$\Theta$ converges to a fixed, deterministic kernel")
is precisely the claim that this spread shrinks toward 0 as width grows — not that the mean
happens to move by a small amount at some particular pair of widths sampled along the way. In
fact, computing the means directly shows the jump from width=10 to width=100 ($\approx0.25$) is
larger than the jump from width=1000 to width=5000 ($\approx0.03$), so even the mean-jump
heuristic points the same direction here — but that agreement is incidental to a small number of
seeds, whereas the shrinking-standard-deviation evidence (roughly a 13x reduction from width=10
to width=5000 in the notebook's own numbers) is the actually load-bearing evidence for
convergence.

</details>

### ✏️ DERIVE
3. In the lazy-training regime, the linearized model's training dynamics under gradient flow
   satisfy $\frac{d}{dt}f_t(\mathbf{x}) = -\Theta(\mathbf{x}, X_{\text{train}})\,
   \big(f_t(X_{\text{train}}) - \mathbf{y}\big)$ for any point $\mathbf{x}$ (train or test), with
   $\Theta$ held fixed. Derive the closed-form fixed point $f_\infty(\mathbf{x})$ that this
   dynamics converges to as $t\to\infty$ (assume $\Theta(X_\text{train},X_\text{train})$ is
   invertible), and show it matches the notebook's `ntk_prediction` formula.

<details>
<summary>💡 Solution outline</summary>

Write $r(t) = f_t(X_\text{train}) - \mathbf{y}$ (the training residual) and
$\Theta_{tt}=\Theta(X_\text{train},X_\text{train})$. Restricting the dynamics to the training
points gives a *closed*, linear ODE in $r$ alone: $\dot r = \dot f_t(X_\text{train}) =
-\Theta_{tt}\,r$, whose solution is $r(t) = e^{-\Theta_{tt}t}\,r(0)$. Since $\Theta_{tt}$ is PSD
(a genuine Gram matrix, per `11.02`), $e^{-\Theta_{tt}t}\to 0$ as $t\to\infty$ (assuming no zero
eigenvalues, i.e. invertibility), so $r(t)\to 0$: the training loss goes to zero, as expected.
For a general point $\mathbf{x}$ (in particular, a test point), integrate its own dynamics using
this now-known $r(t)$:
$$f_\infty(\mathbf{x}) = f_0(\mathbf{x}) - \Theta(\mathbf{x},X_\text{train})\int_0^\infty r(t)\,dt
= f_0(\mathbf{x}) - \Theta(\mathbf{x},X_\text{train})\int_0^\infty e^{-\Theta_{tt}t}r(0)\,dt.$$
The integral evaluates in closed form via $\int_0^\infty e^{-At}\,dt = A^{-1}$ (for PD $A$):
$$f_\infty(\mathbf{x}) = f_0(\mathbf{x}) - \Theta(\mathbf{x},X_\text{train})\,\Theta_{tt}^{-1}\,
r(0) = f_0(\mathbf{x}) + \Theta(\mathbf{x},X_\text{train})\,\Theta_{tt}^{-1}\,\big(\mathbf{y} -
f_0(X_\text{train})\big),$$
using $r(0)=f_0(X_\text{train})-\mathbf{y}$. This is exactly `f_test_init + Theta_te_tr @ alpha`
with `alpha = Theta_tt^{-1} @ (y_train - f_train_init)`, matching the notebook's implementation
verbatim (its `1e-6` ridge term is only a numerical-conditioning safeguard for near-singular
$\Theta_{tt}$, not a change to the underlying formula). A direct numerical check — simulating the
ODE for $r(t)$ via small-step Euler integration until $r(t)\approx 0$, then comparing the
resulting $f_\infty(\mathbf{x})$ to this closed form — confirms agreement to around $10^{-11}$.

</details>

### 🐍 IMPLEMENT
4. The notebook measures "laziness" via the relative change in $\Theta$ before vs. after
   training. Implement a second, independent laziness measure: the relative error between the
   NTK's closed-form prediction (`ntk_prediction`, from the third Implementation check) and what
   the network actually outputs after real training, evaluated across several widths (e.g. 100,
   300, 1000, 3000) using the *same* width for both measures.

<details>
<summary>✅ How to know you're right</summary>

The two independently-computed laziness measures — relative $\Theta$ change, and relative error
between the NTK-predicted and actually-trained output — should agree in *trend* across widths
even though they're computed from different quantities: both should decrease monotonically (or
very close to it) as width increases from 100 to 3000. Concretely, in one run, relative $\Theta$
change fell from about $0.22$ (width 100) to about $0.008$ (width 3000), while the
prediction-vs-actual relative error fell from about $0.53$ to about $0.13$ over the same widths
— different absolute scales (the second measure aggregates two separate sources of
finite-width error, not just $\Theta$'s own drift), but the same qualitative story. If your
version shows one measure improving with width while the other gets worse, that is a sign of a
bug (e.g. reusing a `retain_graph`d computation graph incorrectly, or comparing $\Theta$ at the
wrong training checkpoint), not a genuine finding.

</details>

### 🤖 APPLY
5. Bagging/ensembling is usually motivated as a way to reduce variance from a model's
   sensitivity to its random initialization or training run. Train several independent copies
   of `make_ntk_mlp` (same architecture, different seeds) on the same sine-fitting task, once at
   a narrow width and once at a wide width, and measure how much the trained models' *predictions*
   (not $\Theta$ itself this time) vary across seeds on a shared test grid.

<details>
<summary>✅ What you should observe</summary>

The wide network's predictions should vary far less across random seeds than the narrow
network's — e.g., in one run, the average standard deviation of predictions across 5 seeds
(evaluated pointwise across the test grid) was roughly $0.10$ for `width=20` versus roughly
$0.01$ for `width=3000`, about an order of magnitude tighter. This is the practical, predictive
consequence of the lazy-training story: since a wide network's training dynamics are governed by
a nearly-deterministic, initialization-independent $\Theta$, *what function it ends up computing*
after training is also nearly initialization-independent — ensembling many wide-network runs
buys little beyond what a single run already gives you, whereas ensembling narrow, genuinely
feature-learning networks (which land in meaningfully different places depending on the seed)
can average away real between-run variance.

</details>

### 🚀 CHALLENGE
6. The AI section's overparameterized-but-generalizes-well comparison used $n_\text{train}=6$
   training points. Investigate what happens as you increase $n_\text{train}$ (keep the points
   evenly spaced across $[-2,2]$ and re-fit a degree-$(n_\text{train}-1)$ polynomial each time,
   alongside the NTK-implied interpolator at the same points). Does the polynomial
   interpolator's generalization gap versus the NTK-implied interpolator shrink or grow as
   $n_\text{train}$ increases? Investigate *why*, and connect it to a phenomenon named elsewhere
   in this curriculum's numerical methods material.

<details>
<summary>🔍 What a strong answer covers</summary>

The gap grows, and dramatically: test RMSE for the naive polynomial interpolator explodes as
$n_\text{train}$ increases (in one run: from about $0.49$ at $n_\text{train}=6$ to over $300{,}000$
at $n_\text{train}=25$), while the NTK-implied interpolator's test RMSE stays flat and small
(around $0.11$-$0.13$ throughout) — even though *both* interpolators keep fitting the training
points almost exactly the entire time (both training errors stay at or below $10^{-7}$). A strong
answer identifies this as **Runge's phenomenon**: a high-degree polynomial forced through many
evenly-spaced points develops enormous, uncontrolled oscillations *between* the points, especially
near the domain's edges (in one run, the polynomial's magnitude near the edges of $[-2,2]$ reached
over 2.5 million at $n_\text{train}=25$) — exact interpolation is a much weaker guarantee than it
sounds, since it says nothing about behavior between the fitted points. The NTK-implied
interpolator avoids this entirely because it is the *minimum-RKHS-norm* interpolator, not merely
*an* interpolator — the implicit regularization from gradient descent's own dynamics (this
notebook's central claim) is precisely what prevents the wild between-points behavior that an
unregularized exact fit is free to produce. A strong answer also notes the double meaning of
"more data": more training points make both problems better-posed in principle, but only the
NTK-implied interpolator's inductive bias actually converts that extra data into better
generalization here.

</details>

---

## 📚 Further Reading
- Jacot, Gabriel & Hongler, [\"Neural Tangent Kernel: Convergence and Generalization in Neural Networks\"](https://arxiv.org/abs/1806.07572)
- Lee et al., [\"Wide Neural Networks of Any Depth Evolve as Linear Models Under Gradient Descent\"](https://arxiv.org/abs/1902.06720)
- Belkin, Ma & Mandal, [\"To Understand Deep Learning We Need to Understand Kernel Learning\"](https://arxiv.org/abs/1802.01396)

---

*Next notebook: [Infinite-Width Networks](04_infinite_width_networks.ipynb)*